In [1]:
from edgar import Company, set_identity

set_identity("Kedar Haldar kedar.haldar11@gmail.com")

In [2]:
apple = Company("AAPL")

tenk = apple.get_filings(form="10-K").latest()
xbrl = tenk.xbrl()

bs = xbrl.statements.balance_sheet()
bs_df = bs.to_dataframe()

bs_df.head()

,concept,label,standard_concept,2025-09-27,2024-09-28,level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
0,us-gaap_AssetsAbstract,ASSETS:,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_StatementOfFinancialPositionAbstract
1,us-gaap_AssetsCurrentAbstract,Current assets:,NaN,NaN,NaN,2,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_AssetsAbstract
2,us-gaap_CashAndCashEquivalentsAtCarryingValue,Cash and cash equivalents,CashAndMarketableSecurities,3.593400e+10,2.994300e+10,4,False,False,False,NaN,NaN,NaN,NaN,debit,1.0,1.0,us-gaap_AssetsCurrent,us-gaap_AssetsCurrentAbstract
3,us-gaap_CashAndCashEquivalentsAtCarryingValue,Cash,NaN,2.826700e+10,2.719900e+10,4,False,True,True,us-gaap:FinancialInstrumentAxis,us-gaap_CashMember,Cash,us-gaap:FinancialInstrumentAxis: Cash,debit,1.0,1.0,us-gaap_AssetsCurrent,us-gaap_AssetsCurrentAbstract
4,us-gaap_CashAndCashEquivalentsAtCarryingValue,Level 1 - Money market funds,NaN,5.272000e+09,7.780000e+08,4,False,True,True,us-gaap:FairValueByFairValueHierarchyLevelAxis,us-gaap_FairValueInputsLevel1Member,Level 1,us-gaap:FairValueByFairValueHierarchyLevelAxis...,debit,1.0,1.0,us-gaap_AssetsCurrent,us-gaap_AssetsCurrentAbstract


In [3]:
def get_statement_value(df, standard_concept, period):
    rows = df[df["standard_concept"] == standard_concept]

    if rows.empty:
        return None

    if period not in df.columns:
        return None

    values = rows[period].dropna()

    if values.empty:
        return None

    return values.iloc[0]

In [4]:
def audit_balance_sheet(df, period, tolerance=1e-6):
    assets = get_statement_value(
        df,
        "Assets",
        period
    )

    liabilities = get_statement_value(
        df,
        "Liabilities",
        period
    )

    equity = get_statement_value(
        df,
        "AllEquityBalance",
        period
    )

    reported_total = get_statement_value(
        df,
        "LiabilitiesAndEquity",
        period
    )

    missing = []

    if assets is None:
        missing.append("Assets")

    if liabilities is None:
        missing.append("Liabilities")

    if equity is None:
        missing.append("AllEquityBalance")

    if missing:
        return {
            "period": period,
            "rule": "Assets = Liabilities + Equity",
            "passed": False,
            "status": "missing_data",
            "missing_concepts": missing
        }

    residual = assets - (liabilities + equity)

    result = {
        "period": period,
        "rule": "Assets = Liabilities + Equity",
        "assets": assets,
        "liabilities": liabilities,
        "equity": equity,
        "residual": residual,
        "passed": abs(residual) <= tolerance
    }

    if reported_total is not None:
        result["reported_liabilities_and_equity"] = reported_total
        result["reported_total_residual"] = assets - reported_total

    return result

In [5]:
result = audit_balance_sheet(
    bs_df,
    "2025-09-27"
)

result

{'period': '2025-09-27',
 'rule': 'Assets = Liabilities + Equity',
 'assets': np.float64(359241000000.0),
 'liabilities': np.float64(285508000000.0),
 'equity': np.float64(73733000000.0),
 'residual': np.float64(0.0),
 'passed': np.True_,
 'reported_liabilities_and_equity': np.float64(359241000000.0),
 'reported_total_residual': np.float64(0.0)}

In [6]:
microsoft = Company("MSFT")

msft_10k = microsoft.get_filings(form="10-K").latest()
msft_xbrl = msft_10k.xbrl()

msft_bs = msft_xbrl.statements.balance_sheet()
msft_bs_df = msft_bs.to_dataframe()

msft_bs_df.head()

,concept,label,standard_concept,2026-06-30,2025-06-30,level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
0,us-gaap_AssetsAbstract,Assets,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_StatementOfFinancialPositionAbstract
1,us-gaap_AssetsCurrentAbstract,Current assets:,NaN,NaN,NaN,2,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_AssetsAbstract
2,us-gaap_CashAndCashEquivalentsAtCarryingValue,Cash and cash equivalents,CashAndMarketableSecurities,2.093500e+10,3.024200e+10,4,False,False,False,NaN,NaN,NaN,NaN,debit,1.0,1.0,us-gaap_CashCashEquivalentsAndShortTermInvestm...,us-gaap_AssetsCurrentAbstract
3,us-gaap_CashAndCashEquivalentsAtCarryingValue,Debt Securities - Level 2 - Commercial Paper,NaN,2.373000e+09,9.939000e+09,4,False,True,True,us-gaap:FairValueByAssetClassAxis,us-gaap_DebtSecuritiesMember,Debt Securities,us-gaap:FairValueByAssetClassAxis: Debt Securi...,debit,1.0,1.0,us-gaap_CashCashEquivalentsAndShortTermInvestm...,us-gaap_AssetsCurrentAbstract
4,us-gaap_CashAndCashEquivalentsAtCarryingValue,Debt Securities - Level 2 - Certificates of de...,NaN,1.701000e+09,2.309000e+09,4,False,True,True,us-gaap:FairValueByAssetClassAxis,us-gaap_DebtSecuritiesMember,Debt Securities,us-gaap:FairValueByAssetClassAxis: Debt Securi...,debit,1.0,1.0,us-gaap_CashCashEquivalentsAndShortTermInvestm...,us-gaap_AssetsCurrentAbstract


In [7]:
[
    col for col in msft_bs_df.columns
    if isinstance(col, str) and col[:4].isdigit()
]

['2026-06-30', '2025-06-30']

In [8]:
msft_result = audit_balance_sheet(
    msft_bs_df,
    "2026-06-30"
)

msft_result

{'period': '2026-06-30',
 'rule': 'Assets = Liabilities + Equity',
 'assets': np.float64(758376000000.0),
 'liabilities': np.float64(315989000000.0),
 'equity': np.float64(442387000000.0),
 'residual': np.float64(0.0),
 'passed': np.True_,
 'reported_liabilities_and_equity': np.float64(758376000000.0),
 'reported_total_residual': np.float64(0.0)}

In [9]:
companies = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AMZN": "Amazon",
    "GOOGL": "Alphabet",
    "META": "Meta"
}

In [10]:
def run_company_audit(ticker):
    company = Company(ticker)

    filing = company.get_filings(form="10-K").latest()
    xbrl = filing.xbrl()

    bs = xbrl.statements.balance_sheet()
    df = bs.to_dataframe()

    date_columns = [
        col for col in df.columns
        if isinstance(col, str)
        and len(col) >= 4
        and col[:4].isdigit()
    ]

    latest_period = date_columns[0]

    result = audit_balance_sheet(
        df,
        latest_period
    )

    result["ticker"] = ticker
    return result

In [18]:
run_company_audit("AMZN")

{'period': '2025-12-31',
 'rule': 'Assets = Liabilities + Equity',
 'passed': False,
 'status': 'missing_data',
 'missing_concepts': ['Liabilities'],
 'ticker': 'AMZN'}

In [19]:
amzn = Company("AMZN")

amzn_10k = amzn.get_filings(form="10-K").latest()
amzn_xbrl = amzn_10k.xbrl()

amzn_bs = amzn_xbrl.statements.balance_sheet()
amzn_bs_df = amzn_bs.to_dataframe()

In [20]:
amzn_bs_df[
    amzn_bs_df["label"].str.contains(
        "Liabil",
        case=False,
        na=False
    )
][[
    "concept",
    "label",
    "standard_concept",
    "parent_concept"
]]

,concept,label,standard_concept,parent_concept
27,us-gaap_LiabilitiesAndStockholdersEquityAbstract,LIABILITIES AND STOCKHOLDERS’ EQUITY,NaN,NaN
28,us-gaap_LiabilitiesCurrentAbstract,Current liabilities:,NaN,NaN
32,us-gaap_LiabilitiesCurrent,Total current liabilities,CurrentLiabilitiesTotal,us-gaap_LiabilitiesAndStockholdersEquity
33,amzn_LeaseLiabilityNoncurrent,Long-term lease liabilities,NaN,us-gaap_LiabilitiesAndStockholdersEquity
35,us-gaap_OtherLiabilitiesNoncurrent,Other long-term liabilities,OtherNonOperatingNonCurrentLiabilities,us-gaap_LiabilitiesAndStockholdersEquity
53,us-gaap_LiabilitiesAndStockholdersEquity,Total liabilities and stockholders’ equity,LiabilitiesAndEquity,NaN


In [21]:
amzn_bs_df[
    amzn_bs_df["standard_concept"].astype(str).str.contains(
        "Liabil",
        case=False,
        na=False
    )
][[
    "concept",
    "label",
    "standard_concept"
]]

,concept,label,standard_concept
30,us-gaap_AccruedLiabilitiesCurrent,Accrued expenses and other,OtherOperatingCurrentLiabilities
31,us-gaap_ContractWithCustomerLiabilityCurrent,Unearned revenue,OtherOperatingCurrentLiabilities
32,us-gaap_LiabilitiesCurrent,Total current liabilities,CurrentLiabilitiesTotal
35,us-gaap_OtherLiabilitiesNoncurrent,Other long-term liabilities,OtherNonOperatingNonCurrentLiabilities
53,us-gaap_LiabilitiesAndStockholdersEquity,Total liabilities and stockholders’ equity,LiabilitiesAndEquity


In [22]:
def audit_balance_sheet(df, period, tolerance=1e-6):
    assets = get_statement_value(df, "Assets", period)
    liabilities = get_statement_value(df, "Liabilities", period)
    equity = get_statement_value(df, "AllEquityBalance", period)
    reported_total = get_statement_value(df, "LiabilitiesAndEquity", period)

    results = {
        "period": period
    }

    # Check 1: direct balance-sheet total
    if assets is not None and reported_total is not None:
        residual = assets - reported_total

        results["assets_equals_reported_total"] = {
            "residual": residual,
            "passed": abs(residual) <= tolerance
        }

    # Check 2: accounting equation using separate liabilities + equity
    if assets is not None and liabilities is not None and equity is not None:
        residual = assets - (liabilities + equity)

        results["accounting_equation"] = {
            "residual": residual,
            "passed": abs(residual) <= tolerance
        }
    else:
        missing = []

        if liabilities is None:
            missing.append("Liabilities")
        if equity is None:
            missing.append("Equity")

        results["accounting_equation"] = {
            "status": "missing_data",
            "missing": missing
        }

    return results